In [17]:
import boto3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler


REGION = "us-east-2"  
INSTANCE_ID = "i-0372640781068e64b" 

# AWS:
cloudwatch = boto3.client("cloudwatch", region_name=REGION)

#get metric
def get_metric(metric_name):
    response = cloudwatch.get_metric_statistics(
        Namespace="AWS/EC2",
        MetricName=metric_name,
        Dimensions=[{"Name": "InstanceId", "Value": INSTANCE_ID}],
        StartTime=datetime.now(timezone.utc) - timedelta(hours=24),
        EndTime=datetime.now(timezone.utc),
        Period=300,
        Statistics=["Average"]
    )
    return response.get("Datapoints", [])


# Convert to DataFrame 

def to_df(data, col_name):
    if not data:
        print(f"No data for {col_name}")
        return pd.DataFrame(columns=["Timestamp", col_name])

    df = pd.DataFrame(data)

    if "Timestamp" not in df.columns or "Average" not in df.columns:
        print(f"Unexpected format for {col_name}")
        print(df.head())
        return pd.DataFrame(columns=["Timestamp", col_name])

    df = df.sort_values("Timestamp")
    df = df[["Timestamp", "Average"]]
    df.rename(columns={"Average": col_name}, inplace=True)

    return df


cpu_data = get_metric("CPUUtilization")
net_in_data = get_metric("NetworkIn")
net_out_data = get_metric("NetworkOut")

print("CPU datapoints:", len(cpu_data))
print("NetworkIn datapoints:", len(net_in_data))
print("NetworkOut datapoints:", len(net_out_data))

#create df
df_cpu = to_df(cpu_data, "cpu_usage_pct")
df_in = to_df(net_in_data, "network_in_mb")
df_out = to_df(net_out_data, "network_out_mb")

#merge
df = pd.merge(df_cpu, df_in, on="Timestamp", how="outer")
df = pd.merge(df, df_out, on="Timestamp", how="outer")

# Fill missing values
df = df.sort_values("Timestamp").reset_index(drop=True)
df = df.ffill().bfill()


if "network_in_mb" in df:
    df["network_in_mb"] /= (1024 * 1024)

if "network_out_mb" in df:
    df["network_out_mb"] /= (1024 * 1024)



if len(df) < 10:
    raise ValueError("Not enough data points. Wait a few minutes for CloudWatch metrics.")


# Z-SCORE ANOMALY DETECTION

df["z_score"] = np.abs(stats.zscore(df["cpu_usage_pct"]))
df["is_anomaly"] = (df["z_score"] > 2.5).astype(int)
print(df["is_anomaly"].value_counts())


# PREPARE ML DATA

X = df[[
    "cpu_usage_pct",
    "network_in_mb",
    "network_out_mb"
    
]].values

y = df["is_anomaly"].values


# TRAIN-TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=42, stratify=y
) 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"\nTraining samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")
print(f"Anomalies in training set: {y_train.sum()}")
print(f"Anomalies in test set    : {y_test.sum()}")
print("Anomaly distribution:\n", df["is_anomaly"].value_counts())
print("Newest timestamp:", df["Timestamp"].max())


# TRAIN MODEL

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)



# PREDICT

y_pred = model.predict(X_test)


# EVALUATE

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"\nModel Accuracy: {accuracy * 100:.2f}%")
print("\nConfusion Matrix:")
print(cm)


# FEATURE IMPORTANCE

feature_names = ["cpu_usage_pct", "network_in_mb", "network_out_mb"]

print("\nFeature Importance:")
for name, coef in zip(feature_names, model.coef_[0]):
    print(f"{name:25s}: {coef:.6f}")


print(df.tail()) 
print("Total datapoints:", len(df))
print("\nCPU stats:")
print(df["cpu_usage_pct"].describe())
print("\nZ-score stats:")
print(df["z_score"].describe())
print("\nValue counts:")
print(df["is_anomaly"].value_counts())

CPU datapoints: 82
NetworkIn datapoints: 82
NetworkOut datapoints: 82
is_anomaly
0    80
1     2
Name: count, dtype: int64

Training samples : 41
Testing samples  : 41
Anomalies in training set: 1
Anomalies in test set    : 1
Anomaly distribution:
 is_anomaly
0    80
1     2
Name: count, dtype: int64
Newest timestamp: 2026-03-28 04:17:00+00:00

Model Accuracy: 100.00%

Confusion Matrix:
[[40  0]
 [ 0  1]]

Feature Importance:
cpu_usage_pct            : 1.411181
network_in_mb            : -0.086582
network_out_mb           : -0.059665
                   Timestamp  cpu_usage_pct  network_in_mb  network_out_mb  \
77 2026-03-28 03:57:00+00:00       0.338408       0.055111        0.036760   
78 2026-03-28 04:02:00+00:00       0.261603       0.042072        0.027483   
79 2026-03-28 04:07:00+00:00       1.228309       0.030249        0.024112   
80 2026-03-28 04:12:00+00:00       0.899809       0.043413        0.030420   
81 2026-03-28 04:17:00+00:00       1.291776       0.092415        0.16

/home/ec2-user/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


<bound method Series.max of 0     0.197497
1     0.201411
2     0.204698
3     0.119433
4     0.686784
        ...   
75    0.190332
76    0.191568
77    0.179860
78    0.189314
79    0.006875
Name: z_score, Length: 80, dtype: float64>